In [1]:
import mlflow
import numpy as np
import plotly.graph_objects as go
import torch
import torch.nn as nn
import torch.optim as optim

device = "cuda" if torch.cuda.is_available() else "cpu"
mlflow.set_tracking_uri("../mlruns")
mlflow.set_experiment("pinn")

/home/jean/projects/PINN/.venv/lib/python3.12/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


<Experiment: artifact_location='/home/jean/projects/PINN/notebooks/../mlruns/813368188259171279', creation_time=1775692600536, experiment_id='813368188259171279', last_update_time=1775692600536, lifecycle_stage='active', name='pinn', tags={}, workspace='default'>

Burgers' equation 1D

$$u_t + u\,u_x = \nu\,u_{xx}$$
$$u(x, 0) = -sin(\pi x)$$
$$u(-1, t) = 0$$
$$u(1, t) = 0$$

In [2]:
x_f = np.random.uniform(-1, 1, 10000)
t_f = np.random.uniform(0, 1, 10000)
x_i = np.random.uniform(-1, 1, 200)
t_i = np.zeros_like(x_i)
u_i = -np.sin(np.pi * x_i)
t_b = np.random.uniform(0, 1, 200)
x_b1 = -np.ones_like(t_b)
x_b2 = np.ones_like(t_b)
u_b = np.zeros_like(t_b)

In [3]:
def calculate_grad(
        f: torch.Tensor,
        x: torch.Tensor,
        retain_graph: bool = True
) -> torch.Tensor:
    return  torch.autograd.grad(
            f, x,
            grad_outputs=torch.ones_like(f),
            retain_graph=retain_graph,
            create_graph=True
        )[0]

In [4]:
class BurgersPINN(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.register_buffer("nu", torch.tensor(0.01 / torch.pi))
        self.input = nn.Sequential(
            nn.Linear(2, 50),
            nn.Tanh()
        )

        hidden_layers = []
        for _ in range(8):
            hidden_layers.append(nn.Linear(50, 50))
            hidden_layers.append(nn.Tanh())
        self.hidden = nn.Sequential(*hidden_layers)

        self.output = nn.Sequential(
            nn.Linear(50, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.input(x)
        x = self.hidden(x)
        x = self.output(x)
        return x
    
    def pde_residual(self, x_f: torch.Tensor, t_f: torch.Tensor) -> torch.Tensor:
        u_f = self.forward(torch.cat([x_f, t_f], dim=1))
        u_x = calculate_grad(u_f, x_f)
        u_t = calculate_grad(u_f, t_f)
        u_xx = calculate_grad(u_x, x_f)
        residual = u_t + u_f * u_x - self.nu * u_xx
        return residual
        
        
class MLflowCallback:
    def __init__(self) -> None:
        pass
    
    def on_epoch_end(self, epoch: int, losses_dict: dict[str, float]) -> None:
        for name, value in losses_dict.items():
            mlflow.log_metric(name, value, step=epoch)

    def log_params(self, params_dict: dict[str, int | float]) -> None:
        for name, value in params_dict.items():
            mlflow.log_param(name, value)

In [5]:
model = BurgersPINN().to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=100, 
    min_lr=1e-6
)
epochs = 2000

In [6]:
x_f = torch.tensor(x_f, dtype=torch.float32, requires_grad=True).reshape(-1, 1).to(device)
t_f = torch.tensor(t_f, dtype=torch.float32, requires_grad=True).reshape(-1, 1).to(device)
x_i = torch.tensor(x_i, dtype=torch.float32).reshape(-1, 1).to(device)
t_i = torch.tensor(t_i, dtype=torch.float32).reshape(-1, 1).to(device)
u_i = torch.tensor(u_i, dtype=torch.float32).reshape(-1, 1).to(device)
t_b = torch.tensor(t_b, dtype=torch.float32).reshape(-1, 1).to(device)
x_b1 = torch.tensor(x_b1, dtype=torch.float32).reshape(-1, 1).to(device)
x_b2 = torch.tensor(x_b2, dtype=torch.float32).reshape(-1, 1).to(device)
u_b = torch.tensor(u_b, dtype=torch.float32).reshape(-1, 1).to(device)

In [7]:
with mlflow.start_run():
    callback = MLflowCallback()
    params = {
        "lr": 1e-3,
        "epochs": epochs,
        "nu": model.nu
    }
    callback.log_params(params)
    for epoch in range(epochs):
        optimizer.zero_grad()

        pred_u_i = model(torch.cat([x_i, t_i], dim=1))
        loss_ic = criterion(pred_u_i, u_i)

        pred_u_b1 = model(torch.cat([x_b1, t_b], dim=1))
        pred_u_b2 = model(torch.cat([x_b2, t_b], dim=1))
        loss_bc = criterion(pred_u_b1, u_b) + criterion(pred_u_b2, u_b)

        residual = model.pde_residual(x_f, t_f)
        loss_pde = torch.mean(residual**2)

        loss = 10 * loss_ic + 10 * loss_bc + loss_pde
        losses = {
            "loss_ic": loss_ic.item(),
            "loss_bc": loss_bc.item(),
            "loss_pde": loss_pde.item(),
            "loss": loss.item(),
        }
        callback.on_epoch_end(epoch, losses)
        loss.backward()
        optimizer.step()
        
        scheduler.step(loss.item())

/home/jean/projects/PINN/.venv/lib/python3.12/site-packages/torch/autograd/graph.py:841: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:270.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


In [ ]:
x = np.linspace(-1, 1, 250)
t = np.linspace(0, 1, 250)
x_mesh, t_mesh = np.meshgrid(x, t)
mesh = np.stack([x_mesh.flatten(), t_mesh.flatten()], axis=1)
mesh = torch.tensor(mesh, dtype=torch.float32).to(device)

In [ ]:
model.eval()
with torch.no_grad():
    u_pred = model(mesh).cpu().numpy().reshape(250, 250)

In [23]:
fig = go.Figure()
fig.add_trace(
    go.Heatmap(
        x=x,
        y=t,
        z=u_pred,
        colorscale="Viridis",
        colorbar=dict(title="u(x, t)"),
        zsmooth="best"
    )
)

fig.update_layout(
    title="PINN learned solution",
    xaxis_title="x",
    yaxis_title="t",
    width=800,
    height=600
)

fig.show()

In [24]:
fig = go.Figure()
fig.add_trace(go.Surface(x=x, y=t, z=u_pred, showscale=True, colorscale="Viridis"))
fig.update_layout(
    title="PINN learned solution",
    scene = dict(
        xaxis_title='x',
        yaxis_title='t',
        zaxis_title='u(x,t)'
    ),
    width=900,
    height=700
)

fig.show()
